# 02 — Retrieval and baseline ranking

Read `src/ranking/candidates.py` alongside these actual experiment tables. It implements decayed popularity, age-segment popularity, recency, cosine co-occurrence, implicit ALS, and TF-IDF/SVD content retrieval.

In [ ]:
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
REPORT = ROOT / 'reports/hm_sample'
recall = pd.read_csv(REPORT / 'snapshots/2020-09-16/candidate_recall.csv')
recall

In [ ]:
recall.set_index('source')['candidate_recall'].plot.barh(title='Candidate recall before ranking', xlabel='Macro recall')

## Retrieval versus ordering

Candidate recall counts every target item, including unseen or unretrieved articles. It bounds final Recall@12, but is not itself MAP@12. Candidate quotas are 40 per source and the union averages approximately 147 distinct items. Source baseline lists use historical popularity to fill fewer than 12 recommendations.

In [ ]:
metrics = pd.read_csv(REPORT / 'metrics.csv')
BASELINES = ['popularity', 'segment', 'recent', 'cooccurrence', 'als', 'content']
metrics[(metrics['split'] == 'test') & metrics.model.isin(BASELINES)].set_index('model')

## Read the implementation

- `Retrieval.__init__`: fit each source only on prior history.
- `implicit_als`: alternating confidence-weighted ridge updates, including missing-item terms.
- `Retrieval.generate`: union, source flags, scores and ranks.
- `snapshot` in pipeline.py: measure candidate recall before sampling negatives.

Why does recent-purchase ranking beat global popularity despite low repeat-purchase frequency? Look at the few repeat targets: they are comparatively easy to put near the top of a list, while novel targets spread over a large catalog.

In [ ]:
import json
json.loads((REPORT / 'snapshots/2020-09-16/summary.json').read_text())